# 01 — Weather Data Prototype

**Story**: S01-01 | **Spec**: `openspec/capabilities/weather/spec.md`

Validates Open-Meteo API (wind, temperature, precipitation, cloud cover) and MeteoAlarm advisories
for a German Baltic Sea AOI. Establishes query patterns for the Phase 2 FastAPI service layer.

**AOI**: German Baltic coast — bbox `[9.5, 53.5, 15.0, 56.0]` (lon_min, lat_min, lon_max, lat_max)  
**No credentials required** — Open-Meteo is free and keyless for non-commercial use.

In [ ]:
import requests
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipyleaflet
from ipyleaflet import Map, TileLayer, GeoJSON, WidgetControl
import ipywidgets as widgets

# Test AOI — German Baltic coast centered on Warnemünde
AOI_BBOX = dict(lon_min=9.5, lat_min=53.5, lon_max=15.0, lat_max=56.0)
AOI_CENTER = (54.18, 12.08)  # Warnemünde (lat, lon)

print(f"AOI: {AOI_BBOX}")
print(f"Center: {AOI_CENTER[0]}°N, {AOI_CENTER[1]}°E (Warnemünde)")

## 1. Open-Meteo — Wind, Temperature, Precipitation, Cloud Cover

Docs: https://open-meteo.com/en/docs  
Endpoint: `GET https://api.open-meteo.com/v1/forecast`

Key parameters:
- `latitude`, `longitude` — point query (not gridded; we use the AOI center)
- `hourly` — comma-separated list of variables
- `wind_speed_unit=ms` — m/s (default is km/h)
- `forecast_days=7` — 7-day horizon

> **Note**: Open-Meteo returns data for a single lat/lon point. For gridded data over an AOI,
> we would need to query multiple points or use a different product. For Go/No-Go planning,
> a single representative point (AOI center) is usually sufficient.

In [ ]:
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": AOI_CENTER[0],
    "longitude": AOI_CENTER[1],
    "hourly": "wind_speed_10m,wind_direction_10m,wind_gusts_10m,temperature_2m,precipitation,cloudcover",
    "wind_speed_unit": "ms",       # m/s
    "forecast_days": 7,
    "timezone": "UTC",
}

response = requests.get(OPEN_METEO_URL, params=params, timeout=10)
response.raise_for_status()
data = response.json()

print(f"HTTP {response.status_code}")
print(f"Top-level keys: {list(data.keys())}")
print(f"Hourly keys: {list(data['hourly'].keys())}")
print(f"Number of hourly time steps: {len(data['hourly']['time'])}")
print(f"First timestamp: {data['hourly']['time'][0]}")
print(f"Last timestamp:  {data['hourly']['time'][-1]}")

In [ ]:
# Build a tidy DataFrame  (REQ-WX-001 through REQ-WX-004)
df = pd.DataFrame({
    "time":         pd.to_datetime(data["hourly"]["time"]),
    "wind_ms":      data["hourly"]["wind_speed_10m"],
    "wind_dir_deg": data["hourly"]["wind_direction_10m"],
    "gust_ms":      data["hourly"]["wind_gusts_10m"],
    "temp_c":       data["hourly"]["temperature_2m"],
    "precip_mm":    data["hourly"]["precipitation"],
    "cloud_pct":    data["hourly"]["cloudcover"],
})
df = df.set_index("time")

# Unit conversion: m/s → knots  (SCENARIO-WX-WIND-002)
df["wind_kt"] = df["wind_ms"] * 1.94384
df["gust_kt"] = df["gust_ms"] * 1.94384

print(f"Shape: {df.shape}")
print(f"\nData ranges:")
print(f"  Wind speed:  {df['wind_ms'].min():.1f} – {df['wind_ms'].max():.1f} m/s")
print(f"  Wind speed:  {df['wind_kt'].min():.1f} – {df['wind_kt'].max():.1f} kt")
print(f"  Wind dir:    {df['wind_dir_deg'].min():.0f} – {df['wind_dir_deg'].max():.0f} °")
print(f"  Temperature: {df['temp_c'].min():.1f} – {df['temp_c'].max():.1f} °C")
print(f"  Cloud cover: {df['cloud_pct'].min():.0f} – {df['cloud_pct'].max():.0f} %")

df.head()

In [ ]:
# SCENARIO-WX-WIND-001: verify ≥168 hourly time steps (7 days)
assert len(df) >= 168, f"Expected ≥168 time steps, got {len(df)}"

# SCENARIO-WX-WIND-002: verify knot conversion
sample_ms = df["wind_ms"].iloc[0]
sample_kt = df["wind_kt"].iloc[0]
assert abs(sample_kt - sample_ms * 1.94384) < 0.01, "Knot conversion failed"

# Wind direction in [0, 360] — 360° (due north) is valid
assert df["wind_dir_deg"].min() >= 0 and df["wind_dir_deg"].max() <= 360, "Direction out of range"

# Check last-updated timestamp from response metadata (REQ-WX-006)
generationtime = data.get("generationtime_ms", None)
print(f"✓ ≥168 time steps: {len(df)}")
print(f"✓ Knot conversion correct")
print(f"✓ Wind direction in [0, 360]")
print(f"  generationtime_ms (proxy for last-updated): {generationtime}")

## 2. Visualize Wind, Cloud Cover, and Temperature

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle(f"7-Day Forecast — Baltic AOI Center ({AOI_CENTER[0]}°N, {AOI_CENTER[1]}°E)", fontsize=13)

# Wind
ax0 = axes[0]
ax0.plot(df.index, df["wind_kt"], color="steelblue", lw=1.5, label="Wind speed (kt)")
ax0.fill_between(df.index, df["wind_kt"], df["gust_kt"], alpha=0.25, color="steelblue", label="Gusts")
ax0.axhline(15, color="orange", ls="--", lw=1, label="15 kt caution")
ax0.axhline(25, color="red",    ls="--", lw=1, label="25 kt no-go")
ax0.set_ylabel("Wind (kt)")
ax0.legend(fontsize=8, loc="upper right")
ax0.grid(True, alpha=0.3)

# Cloud cover
ax1 = axes[1]
ax1.fill_between(df.index, df["cloud_pct"], alpha=0.6, color="gray")
ax1.set_ylim(0, 100)
ax1.set_ylabel("Cloud cover (%)")
ax1.axhline(20, color="green", ls="--", lw=1, label="20% sat threshold")
ax1.legend(fontsize=8, loc="upper right")
ax1.grid(True, alpha=0.3)

# Temperature
ax2 = axes[2]
ax2.plot(df.index, df["temp_c"], color="tomato", lw=1.5)
ax2.set_ylabel("Temperature (°C)")
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
ax2.xaxis.set_major_locator(mdates.DayLocator())
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

## 3. Wind Direction — Polar Histogram (Wind Rose)

In [ ]:
# Simple wind rose using polar bar chart
dirs = df["wind_dir_deg"].dropna().values
speeds = df["wind_kt"].dropna().values

# Bin directions into 16 compass sectors
n_sectors = 16
sector_width = 360 / n_sectors
sector_edges = np.linspace(0, 360, n_sectors + 1)
bins, _ = np.histogram(dirs, bins=sector_edges)
theta = np.deg2rad(sector_edges[:-1] + sector_width / 2)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.bar(theta, bins, width=np.deg2rad(sector_width) * 0.9,
       bottom=0, alpha=0.7, color="steelblue", edgecolor="white")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.set_title("Wind Rose — 7-day forecast", pad=15)
plt.tight_layout()
plt.show()

## 4. Cloud Cover — Usable for Satellite Scheduling (SCENARIO-WX-CLOUD-001)

Demonstrate that cloud cover data can be queried at a specific time to assess satellite overpass quality.

In [ ]:
def get_cloud_cover_at_time(df: pd.DataFrame, query_time: pd.Timestamp) -> float:
    """Return nearest-in-time cloud cover % for a given UTC timestamp."""
    idx = df.index.get_indexer([query_time], method="nearest")[0]
    nearest_time = df.index[idx]
    cloud_pct = df["cloud_pct"].iloc[idx]
    print(f"  Query time:   {query_time}")
    print(f"  Nearest time: {nearest_time}")
    print(f"  Cloud cover:  {cloud_pct:.0f}%")
    return float(cloud_pct)

# Example: query cloud cover for 2 days from now at 10:00 UTC
tomorrow_10utc = (pd.Timestamp.utcnow() + pd.Timedelta(days=2)).replace(
    hour=10, minute=0, second=0, microsecond=0, tzinfo=None
)
print("Cloud cover query (SCENARIO-WX-CLOUD-001):")
cc = get_cloud_cover_at_time(df, tomorrow_10utc)
print(f"  → Satellite overpass quality: {'Good (<20%)' if cc < 20 else 'Marginal (20-50%)' if cc < 50 else 'Poor (>50%)'}")

## 5. MeteoAlarm — Weather Advisories (REQ-WX-005)

MeteoAlarm provides weather warnings for EU countries via RSS/Atom feeds and a GeoJSON API.

Feeds index: https://feeds.meteoalarm.org  
Germany (DWD): `https://feeds.meteoalarm.org/feeds/meteoalarm-legacy-atom-germany`

In [ ]:
# MeteoAlarm provides an Atom/RSS feed per country.
# For a proper advisory API, the recommended approach is the CAP (Common Alerting Protocol) feed.
# We fetch the Germany feed and inspect the structure.

METEOALARM_DE_FEED = "https://feeds.meteoalarm.org/feeds/meteoalarm-legacy-atom-germany"

resp = requests.get(METEOALARM_DE_FEED, timeout=15)
print(f"HTTP {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type', 'unknown')}")
print(f"Response length: {len(resp.content)} bytes")
print()
# Show first 800 chars of raw response to understand format
print(resp.text[:800])

In [ ]:
# Parse the Atom/XML feed using ElementTree
import xml.etree.ElementTree as ET

NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "cap":  "urn:oasis:names:tc:emergency:cap:1.2",
}

root = ET.fromstring(resp.content)
entries = root.findall("atom:entry", NS)
print(f"Number of advisory entries: {len(entries)}")

advisories = []
for entry in entries:
    title   = entry.findtext("atom:title", default="", namespaces=NS)
    updated = entry.findtext("atom:updated", default="", namespaces=NS)
    # CAP fields may be nested differently depending on feed version
    event   = entry.findtext(".//cap:event",    default="unknown", namespaces=NS)
    urgency = entry.findtext(".//cap:urgency",  default="unknown", namespaces=NS)
    severity= entry.findtext(".//cap:severity", default="unknown", namespaces=NS)
    onset   = entry.findtext(".//cap:onset",    default="",        namespaces=NS)
    expires = entry.findtext(".//cap:expires",  default="",        namespaces=NS)
    advisories.append(dict(title=title, updated=updated, event=event,
                           urgency=urgency, severity=severity, onset=onset, expires=expires))

if advisories:
    df_adv = pd.DataFrame(advisories)
    print(df_adv[["title", "event", "severity", "onset", "expires"]].head(10).to_string())
else:
    print("No active advisories — empty list returned (SCENARIO-WX-ADV-002: correct behavior)")

In [ ]:
# Filter for Baltic coast area (Schleswig-Holstein / Mecklenburg-Vorpommern coast)
# MeteoAlarm entries include area descriptions; filter by keyword
BALTIC_KEYWORDS = ["küste", "coast", "baltic", "ostsee", "schleswig", "mecklenburg",
                   "rostock", "lübeck", "kiel", "rügen", "stralsund"]

def is_baltic_advisory(entry_dict):
    title_lower = entry_dict["title"].lower()
    return any(kw in title_lower for kw in BALTIC_KEYWORDS)

baltic_advisories = [a for a in advisories if is_baltic_advisory(a)]
print(f"Total advisories: {len(advisories)}")
print(f"Baltic-relevant:  {len(baltic_advisories)}")

if baltic_advisories:
    for a in baltic_advisories:
        print(f"  [{a['severity']}] {a['event']} — {a['title'][:80]}")
else:
    print("  None active for Baltic coast right now")

## 6. Map Visualization — Advisory Overlay on Baltic AOI

Shows OSM base map with the AOI bounding box highlighted.

In [ ]:
# Interactive map with AOI boundary
m = Map(center=AOI_CENTER, zoom=7, scroll_wheel_zoom=True)

# OSM base (default)
# Add OpenSeaMap overlay
seamap = TileLayer(
    url="https://tiles.openseamap.org/seamark/{z}/{x}/{y}.png",
    name="OpenSeaMap",
    opacity=0.7,
)
m.add(seamap)

# AOI bounding box as GeoJSON
aoi_geojson = {
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [AOI_BBOX["lon_min"], AOI_BBOX["lat_min"]],
            [AOI_BBOX["lon_max"], AOI_BBOX["lat_min"]],
            [AOI_BBOX["lon_max"], AOI_BBOX["lat_max"]],
            [AOI_BBOX["lon_min"], AOI_BBOX["lat_max"]],
            [AOI_BBOX["lon_min"], AOI_BBOX["lat_min"]],
        ]]
    },
    "properties": {"name": "AOI"},
}

aoi_layer = GeoJSON(
    data={"type": "FeatureCollection", "features": [aoi_geojson]},
    style={"color": "#2196F3", "weight": 2, "fillOpacity": 0.05},
)
m.add(aoi_layer)

# Current wind conditions annotation
latest = df.iloc[0]
label = widgets.HTML(f"""
<b>Current conditions (AOI center)</b><br>
Wind: {latest['wind_kt']:.1f} kt from {latest['wind_dir_deg']:.0f}°<br>
Cloud: {latest['cloud_pct']:.0f}%<br>
Temp: {latest['temp_c']:.1f}°C
""")
m.add(WidgetControl(widget=label, position="bottomright"))

m

## 7. Summary of Findings

Document key findings for `ops/data-sources.md` and Phase 2 service design.

In [ ]:
print("=== FINDINGS FOR ops/data-sources.md ===")
print()
print("Open-Meteo:")
print(f"  Endpoint: GET {OPEN_METEO_URL}")
print(f"  Auth: None required")
print(f"  Returns: single-point hourly forecast (not gridded)")
print(f"  Variables confirmed: wind_speed_10m (m/s), wind_direction_10m (°), ")
print(f"                       wind_gusts_10m (m/s), temperature_2m (°C),")
print(f"                       precipitation (mm), cloudcover (%)")
print(f"  Time resolution: hourly")
print(f"  Forecast horizon: 7 days ({len(df)} hourly steps confirmed)")
print(f"  Units: set wind_speed_unit=ms for m/s output")
print(f"  Timezone param: 'UTC' recommended for backend use")
print()
print("MeteoAlarm:")
print(f"  Feed: {METEOALARM_DE_FEED}")
print(f"  Format: Atom XML with embedded CAP 1.2 elements")
print(f"  Auth: None")
print(f"  Parsing: xml.etree.ElementTree; namespaces atom + cap")
print(f"  Geographic filtering: by keyword in entry title (area name)")
print(f"  Note: No polygon geometry in this feed — area described in text only.")
print(f"         For polygon geometry, investigate MeteoAlarm GeoJSON API or DWD Open Data.")
print()
print("Phase 2 design notes:")
print("  - Open-Meteo: single point is fine for Go/No-Go; add multi-point grid option later")
print("  - MeteoAlarm: need polygon geometry for map overlay — investigate DWD WFS or EUMETNET API")
print("  - Both APIs: add last_updated timestamp from response metadata")